In [1]:
import os
import cv2
import numpy as np
import random
import shutil
from pathlib import Path
from PIL import Image


# CONFIGURATION

# Original dataset: one folder of single-object images per class (all 224x224)
DATA_DIR = '../data'

# Sorted, so the IDs match the classification notebooks (0 = images_OBJ000, 1 = images_OBJ001, ...)
class_folders = sorted(f for f in os.listdir(DATA_DIR)
                       if os.path.isdir(os.path.join(DATA_DIR, f)))

# Class name = folder name without the 'images_' prefix, e.g. 'OBJ000'
DATASET_PATHS = {f.replace('images_', '').strip(): os.path.join(DATA_DIR, f)
                 for f in class_folders}

# Object IDs (MUST match what you use in training)
OBJECT_IDS = {name: i for i, name in enumerate(DATASET_PATHS)}

# Output folder: one sub-folder per split -> combined_dataset/{train,val,test}/{images,labels}
OUTPUT_DIR = './combined_dataset'

# Split the single-object photos of every class 80 / 10 / 10 BEFORE combining them,
# so a photo used in a train image never appears in a val or test image
SPLIT_RATIOS = {'train': 0.8, 'val': 0.1, 'test': 0.1}
SEED = 42

# Canvas size (background image size)
CANVAS_WIDTH = 640
CANVAS_HEIGHT = 640

# Number of combined images to create per split (500 in total)
NUM_COMBINED_IMAGES = {'train': 400, 'val': 50, 'test': 50}


# AUGMENTATION CONFIGURATION

# Enable/Disable augmentations
USE_AUGMENTATION = True

# Augmentation probabilities (0.0 to 1.0)
AUGMENTATION_PROBS = {
    'rotation': 0.5,      # 50% chance to rotate
    'flip': 0.4,          # 40% chance to flip
    'brightness': 0.3,    # 30% chance to adjust brightness
    'blur': 0.2,          # 20% chance to blur
    'zoom': 0.3           # 30% chance to zoom
}

# Augmentation parameters
ROTATION_RANGE = (-25, 25)        # Rotate between -25 to +25 degrees
BRIGHTNESS_RANGE = (0.7, 1.3)     # Darken to brighten (0.7x to 1.3x)
BLUR_KERNEL_RANGE = (3, 9)        # Blur kernel size (must be odd)
ZOOM_RANGE = (0.8, 1.2)           # Zoom between 0.8x to 1.2x


# CREATE OUTPUT DIRECTORIES
# Delete the old combined_dataset first so old images/labels never mix with the new ones
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
for split in SPLIT_RATIOS:
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'labels'), exist_ok=True)

# Fixed seed, so the split and the combined images are the same every run
random.seed(SEED)


# LOAD ALL IMAGES INTO MEMORY
print("Loading images...")
image_database = {}

for object_name, folder_path in DATASET_PATHS.items():
    image_database[object_name] = []
    
    # Check if folder exists
    if not os.path.exists(folder_path):
        print(f"⚠️  Warning: Folder {folder_path} not found!")
        continue
    
    # Load all images from folder
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            image_path = os.path.join(folder_path, filename)
            try:
                # Load image
                img = cv2.imread(image_path)
                if img is not None:
                    image_database[object_name].append((filename, img))
            except Exception as e:
                print(f"Error loading {image_path}: {e}")
    
    print(f"✓ Loaded {len(image_database[object_name])} images of {object_name}")


# SPLIT EACH CLASS INTO TRAIN / VAL / TEST

split_database = {split: {} for split in SPLIT_RATIOS}
split_rows = ['split,class,filename']

print("\nSplitting single-object photos (train / val / test):")
for object_name, items in image_database.items():
    items = sorted(items, key=lambda item: item[0])   # sort by filename, then shuffle with the seed
    random.shuffle(items)

    n_train = int(len(items) * SPLIT_RATIOS['train'])
    n_val = int(len(items) * SPLIT_RATIOS['val'])
    parts = {'train': items[:n_train],
             'val': items[n_train:n_train + n_val],
             'test': items[n_train + n_val:]}

    for split, part in parts.items():
        split_database[split][object_name] = [img for _, img in part]
        split_rows += [f"{split},{object_name},{filename}" for filename, _ in part]

    print(f"  {object_name}: {len(parts['train'])} / {len(parts['val'])} / {len(parts['test'])}")

# Save which photo went to which split, so the split can be checked later
with open(os.path.join(OUTPUT_DIR, 'source_split.csv'), 'w') as f:
    f.write('\n'.join(split_rows) + '\n')


# AUGMENTATION FUNCTIONS

def rotate_image(image, angle):
    """
    Rotate image by given angle (in degrees)
    Positive angle = counter-clockwise
    """
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    
    # Get rotation matrix
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    
    # Apply rotation
    rotated = cv2.warpAffine(image, rotation_matrix, (w, h))
    
    return rotated

def flip_image(image, direction='horizontal'):
    """
    Flip image horizontally or vertically
    direction: 'horizontal' or 'vertical'
    """
    if direction == 'horizontal':
        flipped = cv2.flip(image, 1)  # 1 = flip horizontally
    else:  # vertical
        flipped = cv2.flip(image, 0)  # 0 = flip vertically
    
    return flipped

def adjust_brightness(image, factor):
    """
    Adjust image brightness
    factor < 1.0 = darker
    factor > 1.0 = brighter
    """
    # Convert to float for calculation
    img_float = image.astype(np.float32)
    
    # Multiply by factor
    adjusted = img_float * factor
    
    # Clip to valid range (0-255)
    adjusted = np.clip(adjusted, 0, 255)
    
    # Convert back to uint8
    adjusted = adjusted.astype(np.uint8)
    
    return adjusted

def blur_image(image, kernel_size):
    """
    Apply Gaussian blur to image
    kernel_size: must be odd number
    """
    # Make sure kernel_size is odd
    if kernel_size % 2 == 0:
        kernel_size += 1
    
    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
    
    return blurred

def zoom_image(image, zoom_factor):
    """
    Zoom image in or out
    zoom_factor < 1.0 = zoom out
    zoom_factor > 1.0 = zoom in
    """
    h, w = image.shape[:2]
    
    # Calculate new dimensions
    new_h = int(h * zoom_factor)
    new_w = int(w * zoom_factor)
    
    # Resize
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # If zoomed in, crop the center
    if zoom_factor > 1.0:
        start_y = (new_h - h) // 2
        start_x = (new_w - w) // 2
        resized = resized[start_y:start_y+h, start_x:start_x+w]
    
    # If zoomed out, pad with white
    else:
        padded = np.ones((h, w, 3), dtype=np.uint8) * 255
        start_y = (h - new_h) // 2
        start_x = (w - new_w) // 2
        padded[start_y:start_y+new_h, start_x:start_x+new_w] = resized
        resized = padded
    
    return resized

def apply_augmentation(image):
    """
    Apply random augmentations to image
    """
    if not USE_AUGMENTATION:
        return image
    
    augmented = image.copy()
    
    # 1. ROTATION
    if random.random() < AUGMENTATION_PROBS['rotation']:
        angle = random.uniform(ROTATION_RANGE[0], ROTATION_RANGE[1])
        augmented = rotate_image(augmented, angle)
        # print(f"   ↻ Rotated by {angle:.1f}°")
    
    # 2. FLIP
    if random.random() < AUGMENTATION_PROBS['flip']:
        direction = random.choice(['horizontal', 'vertical'])
        augmented = flip_image(augmented, direction)
        # print(f"   ↔ Flipped {direction}")
    
    # 3. BRIGHTNESS
    if random.random() < AUGMENTATION_PROBS['brightness']:
        brightness_factor = random.uniform(BRIGHTNESS_RANGE[0], BRIGHTNESS_RANGE[1])
        augmented = adjust_brightness(augmented, brightness_factor)
        # print(f"   ☀ Brightness: {brightness_factor:.2f}x")
    
    # 4. BLUR
    if random.random() < AUGMENTATION_PROBS['blur']:
        kernel_size = random.randint(BLUR_KERNEL_RANGE[0], BLUR_KERNEL_RANGE[1])
        if kernel_size % 2 == 0:  # Make odd
            kernel_size += 1
        augmented = blur_image(augmented, kernel_size)
        # print(f"   ≈ Blurred (kernel={kernel_size})")
    
    # 5. ZOOM
    if random.random() < AUGMENTATION_PROBS['zoom']:
        zoom_factor = random.uniform(ZOOM_RANGE[0], ZOOM_RANGE[1])
        augmented = zoom_image(augmented, zoom_factor)
        # print(f"   🔍 Zoomed: {zoom_factor:.2f}x")
    
    return augmented


# FUNCTION TO RESIZE IMAGE TO FIT

def resize_image_to_fit(image, max_width=150, max_height=150):
    """
    Resize image proportionally to fit within max dimensions
    while keeping aspect ratio
    """
    h, w = image.shape[:2]
    
    # Calculate scaling factor
    scale = min(max_width / w, max_height / h)
    
    new_w = int(w * scale)
    new_h = int(h * scale)
    
    # Resize image
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return resized


# FUNCTION TO PASTE IMAGE

def paste_image_on_canvas(canvas, object_img, x, y):
    """
    Paste an object image on the canvas at position (x, y)
    Returns: updated canvas and bounding box of pasted object
    """
    obj_h, obj_w = object_img.shape[:2]
    
    # Make sure image doesn't go outside canvas
    x = max(0, min(x, CANVAS_WIDTH - obj_w))
    y = max(0, min(y, CANVAS_HEIGHT - obj_h))
    
    # Get the end coordinates
    x_end = x + obj_w
    y_end = y + obj_h
    
    # Handle transparency (if image has alpha channel)
    if object_img.shape[2] == 4:  # Has alpha channel
        alpha = object_img[:, :, 3] / 255.0
        for c in range(3):
            canvas[y:y_end, x:x_end, c] = (
                object_img[:, :, c] * alpha +
                canvas[y:y_end, x:x_end, c] * (1 - alpha)
            )
    else:
        # No transparency, just paste
        canvas[y:y_end, x:x_end] = object_img
    
    # Return bounding box coordinates
    return canvas, (x, y, x_end, y_end)


# FUNCTION TO CONVERT TO YOLO FORMAT

def bbox_to_yolo_format(bbox, canvas_width, canvas_height):
    """
    Convert bounding box (x_min, y_min, x_max, y_max) to YOLO format
    YOLO format: center_x center_y width height (normalized 0-1)
    """
    x_min, y_min, x_max, y_max = bbox
    
    # Calculate center
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0
    
    # Calculate width and height
    width = x_max - x_min
    height = y_max - y_min
    
    # Normalize to 0-1
    center_x_norm = center_x / canvas_width
    center_y_norm = center_y / canvas_height
    width_norm = width / canvas_width
    height_norm = height / canvas_height
    
    return center_x_norm, center_y_norm, width_norm, height_norm


# MAIN FUNCTION: CREATE COMBINED IMAGES

def create_combined_dataset(split, num_images):
    """
    Create combined images with automatic labels and augmentations for one split,
    using only that split's single-object photos
    """
    images_dir = os.path.join(OUTPUT_DIR, split, 'images')
    labels_dir = os.path.join(OUTPUT_DIR, split, 'labels')
    split_images = split_database[split]
    
    print(f"\n{'='*60}")
    print(f"Creating {num_images} combined {split} images...")
    print(f"Augmentation: {'ENABLED' if USE_AUGMENTATION else 'DISABLED'}")
    print(f"{'='*60}\n")
    
    for image_num in range(num_images):
        # Create a blank canvas (white background)
        canvas = np.ones((CANVAS_HEIGHT, CANVAS_WIDTH, 3), dtype=np.uint8) * 255
        
        # List to store labels for this image
        labels = []
        
        # Randomly decide how many objects to place (2-4)
        num_objects = random.randint(2, 4)
        
        # Randomly select which objects to place
        selected_objects = random.sample(list(DATASET_PATHS.keys()), 
                                        min(num_objects, len(DATASET_PATHS)))
        
        # Place each selected object on canvas
        for object_name in selected_objects:
            # Check if we have images for this object
            if not split_images[object_name]:
                print(f"⚠️  No {split} images available for {object_name}")
                continue
            
            # Randomly pick an image of this object (from this split only)
            random_object_img = random.choice(split_images[object_name])
            
            # Convert to 3-channel if needed
            if len(random_object_img.shape) == 2:
                random_object_img = cv2.cvtColor(random_object_img, cv2.COLOR_GRAY2BGR)
            
            # ===== APPLY AUGMENTATION =====
            augmented_img = apply_augmentation(random_object_img)
            
            # Resize the augmented object image to fit on canvas
            resized_obj = resize_image_to_fit(augmented_img, 
                                             max_width=120, 
                                             max_height=120)
            
            # Random position on canvas (but not too close to edges)
            random_x = random.randint(10, CANVAS_WIDTH - resized_obj.shape[1] - 10)
            random_y = random.randint(10, CANVAS_HEIGHT - resized_obj.shape[0] - 10)
            
            # Paste the object on canvas
            canvas, bbox = paste_image_on_canvas(canvas, resized_obj, 
                                                random_x, random_y)
            
            # Convert to YOLO format
            yolo_label = bbox_to_yolo_format(bbox, CANVAS_WIDTH, CANVAS_HEIGHT)
            
            # Store label: object_id center_x center_y width height
            object_id = OBJECT_IDS[object_name]
            labels.append(f"{object_id} {yolo_label[0]:.6f} {yolo_label[1]:.6f} {yolo_label[2]:.6f} {yolo_label[3]:.6f}")
        
        # Save combined image
        image_filename = f"{split}_{image_num:04d}.jpg"
        image_path = os.path.join(images_dir, image_filename)
        cv2.imwrite(image_path, canvas)
        
        # Save labels to text file
        label_filename = f"{split}_{image_num:04d}.txt"
        label_path = os.path.join(labels_dir, label_filename)
        
        with open(label_path, 'w') as f:
            for label in labels:
                f.write(label + '\n')
        
        # Print progress
        if (image_num + 1) % 50 == 0:
            print(f"✓ Created {image_num + 1}/{num_images} combined {split} images")
    
    print(f"\n✅ Successfully created {num_images} combined {split} images!")
    print(f"   Images saved to: {images_dir}")
    print(f"   Labels saved to: {labels_dir}")


# RUN THE SCRIPT

if __name__ == "__main__":
    for split, num_images in NUM_COMBINED_IMAGES.items():
        create_combined_dataset(split, num_images)
    print("\n🎉 Dataset creation complete!")
    print(f"Photo split saved to: {os.path.join(OUTPUT_DIR, 'source_split.csv')}")
    print("You can now use these images for YOLOv8 training\n")


Loading images...
✓ Loaded 100 images of OBJ000
✓ Loaded 100 images of OBJ001
✓ Loaded 100 images of OBJ002
✓ Loaded 100 images of OBJ004
✓ Loaded 100 images of OBJ006


✓ Loaded 100 images of OBJ007
✓ Loaded 100 images of OBJ011
✓ Loaded 100 images of OBJ070
✓ Loaded 100 images of OBJ108
✓ Loaded 100 images of OBJ111
✓ Loaded 100 images of OBJ123


✓ Loaded 100 images of OBJ125
✓ Loaded 100 images of OBJ201
✓ Loaded 100 images of OBJ222
✓ Loaded 100 images of OBJ523
✓ Loaded 114 images of OBJ529
✓ Loaded 108 images of OBJ608


✓ Loaded 100 images of OBJ666
✓ Loaded 100 images of OBJ777
✓ Loaded 100 images of OBJ916

Splitting single-object photos (train / val / test):
  OBJ000: 80 / 10 / 10
  OBJ001: 80 / 10 / 10
  OBJ002: 80 / 10 / 10
  OBJ004: 80 / 10 / 10
  OBJ006: 80 / 10 / 10
  OBJ007: 80 / 10 / 10
  OBJ011: 80 / 10 / 10
  OBJ070: 80 / 10 / 10
  OBJ108: 80 / 10 / 10
  OBJ111: 80 / 10 / 10
  OBJ123: 80 / 10 / 10
  OBJ125: 80 / 10 / 10
  OBJ201: 80 / 10 / 10
  OBJ222: 80 / 10 / 10
  OBJ523: 80 / 10 / 10
  OBJ529: 91 / 11 / 12
  OBJ608: 86 / 10 / 12
  OBJ666: 80 / 10 / 10
  OBJ777: 80 / 10 / 10
  OBJ916: 80 / 10 / 10

Creating 400 combined train images...
Augmentation: ENABLED



✓ Created 50/400 combined train images
✓ Created 100/400 combined train images


✓ Created 150/400 combined train images
✓ Created 200/400 combined train images


✓ Created 250/400 combined train images
✓ Created 300/400 combined train images


✓ Created 350/400 combined train images
✓ Created 400/400 combined train images

✅ Successfully created 400 combined train images!
   Images saved to: ./combined_dataset/train/images
   Labels saved to: ./combined_dataset/train/labels

Creating 50 combined val images...
Augmentation: ENABLED



✓ Created 50/50 combined val images

✅ Successfully created 50 combined val images!
   Images saved to: ./combined_dataset/val/images
   Labels saved to: ./combined_dataset/val/labels

Creating 50 combined test images...
Augmentation: ENABLED

✓ Created 50/50 combined test images

✅ Successfully created 50 combined test images!
   Images saved to: ./combined_dataset/test/images
   Labels saved to: ./combined_dataset/test/labels

🎉 Dataset creation complete!
Photo split saved to: ./combined_dataset/source_split.csv
You can now use these images for YOLOv8 training

